Añadir al graph retrieval:
-   Extraccion de entidades de la pregunta
-   Creacion indice de texto
-   Busqueda exacta
-   Busqueda fuzzy
-   Re-ranking de resultados

INICIALIZACION

In [68]:
import sys
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase

In [2]:
sys.path.append("../src")

In [ ]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j
from graph_retrieval.funciones_graph_retrieval import (
    extraer_top_k_entities,
    formatear_tripletas,
)

from funciones_generales import build_prompt
from LLM_interaction import LLM_interaction_functions as llm_funcs
from metricas.metricas_2Wiki import (
    f1_score,
    exact_match_score,
    respuesta_en_nodos_encontrados,
    suporting_facts_en_subgrafo,
    metricas_totales
    )
from output_save.funciones_guardado import guardar_resultados, guardar_registro

In [8]:
import spacy

# Load Data

In [4]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 100, "train")

In [208]:
ejemplo = dataset_2Wiki[3]

# Qdrant y Neo4j conexion

In [6]:
database_Neo = "2wiki.prueba1"
database_Neo4j = ConexionNeo4j(database_Neo)

In [7]:
collection = "2wikimultihop_prueba1"
embed_model_st = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5051.15it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Extraer entidades (spacy)

In [209]:
dataset_2Wiki[3]

{'_id': '633f80660bdd11eba7f7acde48001122',
 'type': 'compositional',
 'question': "What is the date of birth of Mina Gerhardsen's father?",
 'context': '[["Pamela Jain", ["Pamela Jain is an Indian playback singer.", "Date of Birth:16th March."]], ["Terence Robinson", ["Terence D. Robinson( date of birth and death unknown) was a male wrestler who competed for England."]], ["Mina Gerhardsen", ["Mina Gerhardsen (born 14 September 1975) is a Norwegian politician for the Labour Party.", "She is the daughter of Rune Gerhardsen and Tove Strand, and granddaughter of Einar Gerhardsen.", "She is married to Eirik \\u00d8wre Thorshaug.", "She led the Oslo branch of Natur og Ungdom from 1993 to 1995, and was deputy leader of the Workers\' Youth League in Oslo in 1997.", "She took the cand.mag.", "degree at the University of Oslo in 1998, and also has master\'s degrees in pedagogy from 2000 and human geography from 2003.", "From 1999 to 2002 she worked part-time as a journalist in \\"Dagsavisen\\" 

In [210]:
question = dataset_2Wiki[3]['question']

In [212]:
ner = spacy.load('en_core_web_sm')

In [213]:
entidades = ner(question)

In [214]:
entidades.ents

(Mina Gerhardsen's,)

Añadir entity ruler para que encuentre palabras que empiecen por mayuscula

In [215]:
if "entity_ruler" in ner.pipe_names:
    ner.remove_pipe("entity_ruler")
ner_custom = ner.add_pipe("entity_ruler", before="ner")

patron_mayuscula = [
    {
        "IS_TITLE": True,  # Detecta entidades cuando empiezan por mayuscula
        "OP": "+"          # Se repite 1 o más veces
    }
]
patterns = [
    {
        "label": "MOVIE",          # El nombre de la entidad que quieres asignar
        "pattern": patron_mayuscula  # El patrón que definiste arriba
    }
]


ner_custom.add_patterns(patterns)

In [216]:
print(question)
entidades = ner(question)
print(entidades.ents)

What is the date of birth of Mina Gerhardsen's father?
(What, Mina Gerhardsen)


In [217]:
entidades.ents[0]

What

In [218]:
print(entidades.ents[1].text)
print(entidades.ents[1].label_)
print(entidades.ents[1].start_char)
print(entidades.ents[1].end_char)
print(entidades.ents[1].sent.text)

Mina Gerhardsen
MOVIE
29
44
What is the date of birth of Mina Gerhardsen's father?


Extraer entidades - La primera no porque es la primera palabra que va en mayuscula

In [219]:
entidades_ner = [ent.text for ent in entidades.ents]
entidades_ner = entidades_ner[1:]

In [220]:
entidades_ner

['Mina Gerhardsen']

In [221]:
entidades_ner = list(set(entidades_ner))

In [222]:
entidades_ner

['Mina Gerhardsen']

In [211]:
def extraer_entidades_ner(text):
    if "entity_ruler" in ner.pipe_names:
        ner.remove_pipe("entity_ruler")
    ner_custom = ner.add_pipe("entity_ruler", before="ner")

    patron_mayuscula = [
        {
            "IS_TITLE": True,  # Detecta entidades cuando empiezan por mayuscula
            "OP": "+"          # Se repite 1 o más veces
        }
    ]
    patterns = [
        {
            "label": "NOMBRE",          # El nombre de la entidad que quieres asignar
            "pattern": patron_mayuscula  # El patrón que definiste arriba
        }
    ]


    ner_custom.add_patterns(patterns)
    
    entidades_extraidas = ner(text)
    entidades_ner = [ent.text for ent in entidades_extraidas.ents]
    entidades_ner = entidades_ner[1:]
    
    return list(set(entidades_ner))

In [101]:
entidades_ner = extraer_entidades_ner(question)

In [223]:
entidades_ner

['Mina Gerhardsen']

# Busqueda Exacta entidades encontradas

In [ ]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    database = "2wiki.prueba1"
)

In [ ]:

def busqueda_exacta_entidades(driver, entidades):
    
    query = """
        MATCH (n:Entity)
        WHERE n.name IN $entidades
        RETURN n.name
    """
    # summary = self.driver.execute_query(query, index_name = index_name)
    records, summary, keys = driver.execute_query(query, entidades = entidades)
    entis_exactas = [record["n.name"] for record in records]
    return entis_exactas

In [225]:
entis_exactas = busqueda_exacta_entidades(driver, entidades_ner)
# entis_exactas = busqueda_exacta_entidades(driver, ['Canada'])

In [226]:
entis_exactas

[<Record n.name='Mina Gerhardsen'>]

In [229]:
for record in entis_exactas:
    print(record["n.name"])

Mina Gerhardsen


In [230]:
[record["n.name"] for record in entis_exactas]

['Mina Gerhardsen']

# Creacion de Text Index en Neo4j

In [121]:
def crear_fulltext_index(driver, index_name):
    
    query = """
        CREATE FULLTEXT INDEX $index_name FOR (n:Entity) ON EACH [n.name]
    """
    # summary = self.driver.execute_query(query, index_name = index_name)
    records, summary, keys = driver.execute_query(query, index_name = index_name)
    return records


In [ ]:
index_name = "entidadesIndex"
sumary = crear_fulltext_index(driver, index_name)

In [86]:
sumary

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x0000027CDFAFE590>, keys=[])

# Busqueda al Text Index de entidades extraidas

fulltext-queryNodes --> Busca entidades que contengan ese string  
Si se le añade ~ despues busca parecidos

In [136]:
query = """
CALL db.index.fulltext.queryNodes($index_name, "fiml~") 
YIELD node, score
RETURN node.name, score
"""

In [137]:
records, summary, keys = driver.execute_query(query, index_name = index_name)

In [138]:
records

[<Record node.name='Fire Down Below (1957 film)' score=1.6022485494613647>,
 <Record node.name='Road Kill' score=1.3400148153305054>,
 <Record node.name='Lima Barreto' score=1.3400148153305054>,
 <Record node.name='Blücher (film)' score=1.120843768119812>,
 <Record node.name='Musa (film)' score=1.120843768119812>,
 <Record node.name='Méditerranée (1963 film)' score=0.9509748220443726>,
 <Record node.name='Bait (1954 film)' score=0.9509748220443726>,
 <Record node.name='Pony Express (film)' score=0.9509748220443726>,
 <Record node.name='Timecode (2000 film)' score=0.9509748220443726>,
 <Record node.name='1922 (2017 film)' score=0.9509748220443726>,
 <Record node.name='Move (1970 film)' score=0.9509748220443726>,
 <Record node.name='Glamour Boy (film)' score=0.9509748220443726>,
 <Record node.name='The Falcon (film)' score=0.9509748220443726>,
 <Record node.name='The Cup (2011 film)' score=0.8258184194564819>,
 <Record node.name='Passage West (1951 film)' score=0.8258184194564819>,
 <Rec

In [167]:
def busqueda_parcial_entidades(driver, index_name, entidades):
    
    res_busqueda_parcial = {}
    for ent in entidades:
        query = """
            CALL db.index.fulltext.queryNodes($index_name, $entidad) 
            YIELD node, score
            RETURN node.name, score
        """
        records, summary, keys = driver.execute_query(query, index_name = index_name, entidad = ent)
        resultado = [{"name": record['node.name'], "score": record['score']} for record in records]
        res_busqueda_parcial[ent] = resultado
    return res_busqueda_parcial
    

In [141]:
index_name = "entidadesIndex"

In [168]:
res_busqueda_parcial = busqueda_parcial_entidades(driver, index_name, entidades_ner)

In [169]:
res_busqueda_parcial

{'Move': [{'name': 'Move (1970 film)', 'score': 2.2738592624664307}],
 'Méditerranée': [{'name': 'Méditerranée (1963 film)',
   'score': 2.2738592624664307}],
 'Film': [{'name': 'Musa (film)', 'score': 1.4944583177566528},
  {'name': 'Blücher (film)', 'score': 1.4944583177566528},
  {'name': 'Bait (1954 film)', 'score': 1.2679665088653564},
  {'name': '1922 (2017 film)', 'score': 1.2679665088653564},
  {'name': 'Timecode (2000 film)', 'score': 1.2679665088653564},
  {'name': 'Méditerranée (1963 film)', 'score': 1.2679665088653564},
  {'name': 'Glamour Boy (film)', 'score': 1.2679665088653564},
  {'name': 'The Falcon (film)', 'score': 1.2679665088653564},
  {'name': 'Pony Express (film)', 'score': 1.2679665088653564},
  {'name': 'Move (1970 film)', 'score': 1.2679665088653564},
  {'name': 'The Cup (2011 film)', 'score': 1.1010913848876953},
  {'name': 'Passage West (1951 film)', 'score': 1.1010913848876953},
  {'name': 'Fire Down Below (1957 film)', 'score': 0.973031759262085},
  {'name

In [164]:
def busqueda_fuzzy_entidades(driver, index_name, entidades):
    
    res_busqueda_fuzzy = {}
    for ent in entidades:
        # ent_fuzzy = f"{ent}~"
        ent_fuzzy = ent + "~"
        query = """
            CALL db.index.fulltext.queryNodes($index_name, $entidad) 
            YIELD node, score
            RETURN node.name, score
        """
        records, summary, keys = driver.execute_query(query, index_name = index_name, entidad = ent_fuzzy)
        # res_busqueda_fuzzy[ent] = records
        resultado = [{"name": record['node.name'], "score": record['score']} for record in records]
        res_busqueda_fuzzy[ent] = resultado
    return res_busqueda_fuzzy

In [165]:
res_busqueda_fuzzy = busqueda_fuzzy_entidades(driver, index_name, entidades_ner)

In [166]:
res_busqueda_fuzzy

{'Move': [{'name': 'Move (1970 film)', 'score': 2.2738592624664307},
  {'name': 'The Love Route', 'score': 1.7053942680358887},
  {'name': 'Rome', 'score': 1.6314306259155273},
  {'name': 'The Money Changers', 'score': 1.1369296312332153},
  {'name': 'Avidathe Pole Ivideyum', 'score': 1.1369296312332153},
  {'name': 'Model Technical Higher Secondary Schools',
   'score': 0.8724746704101562},
  {'name': 'Mr. Moto Takes a Chance', 'score': 0.8724746704101562},
  {'name': 'The Da Vinci Code (film)', 'score': 0.8724746704101562},
  {'name': 'Every Day I Have to Cry', 'score': 0.7815757989883423},
  {'name': 'One Hundred Nails', 'score': 0.7579530477523804},
  {'name': 'Charge It to Me', 'score': 0.0}],
 'Méditerranée': [{'name': 'Méditerranée (1963 film)',
   'score': 2.2738592624664307}],
 'Film': [{'name': 'Fire Down Below (1957 film)', 'score': 1.8455064296722412},
  {'name': 'Musa (film)', 'score': 1.4944583177566528},
  {'name': 'Blücher (film)', 'score': 1.4944583177566528},
  {'name

# Busqueda a los embeddings

In [146]:
vector_index_name = "entity_embedding_index"


In [147]:
res_busqueda_embeddings = database_Neo4j.query_a_embedding(vector_index_name, embed_model_st, question, 5)

In [148]:
res_busqueda_embeddings

[<Record name='Méditerranée (1963 film)' score=0.8802040815353394>,
 <Record name='Alsino and the Condor' score=0.7534630298614502>,
 <Record name='Jean-Daniel Pollet' score=0.7435126304626465>,
 <Record name='Escape to France' score=0.7428261637687683>,
 <Record name='1960' score=0.7390859127044678>]

# Filtrado de resultados

Entidades Seleccionadas:
-   Todas las que hay en entis exactas.
-   En parciales, fuzzy y embeddings. 1º filtrado por score, Luego ver las que coinciden. Dejar las mas repetidas
-   

In [149]:
entidades_ner

['Move', 'Méditerranée', 'Film', '1970', '1963']

In [151]:
entis_exactas

[]

In [152]:
res_busqueda_parcial

{'Move': [<Record node.name='Move (1970 film)' score=2.2738592624664307>],
 'Méditerranée': [<Record node.name='Méditerranée (1963 film)' score=2.2738592624664307>],
 'Film': [<Record node.name='Musa (film)' score=1.4944583177566528>,
  <Record node.name='Blücher (film)' score=1.4944583177566528>,
  <Record node.name='Bait (1954 film)' score=1.2679665088653564>,
  <Record node.name='1922 (2017 film)' score=1.2679665088653564>,
  <Record node.name='Timecode (2000 film)' score=1.2679665088653564>,
  <Record node.name='Méditerranée (1963 film)' score=1.2679665088653564>,
  <Record node.name='Glamour Boy (film)' score=1.2679665088653564>,
  <Record node.name='The Falcon (film)' score=1.2679665088653564>,
  <Record node.name='Pony Express (film)' score=1.2679665088653564>,
  <Record node.name='Move (1970 film)' score=1.2679665088653564>,
  <Record node.name='The Cup (2011 film)' score=1.1010913848876953>,
  <Record node.name='Passage West (1951 film)' score=1.1010913848876953>,
  <Record no

In [170]:
res_busqueda_fuzzy

{'Move': [{'name': 'Move (1970 film)', 'score': 2.2738592624664307},
  {'name': 'The Love Route', 'score': 1.7053942680358887},
  {'name': 'Rome', 'score': 1.6314306259155273},
  {'name': 'The Money Changers', 'score': 1.1369296312332153},
  {'name': 'Avidathe Pole Ivideyum', 'score': 1.1369296312332153},
  {'name': 'Model Technical Higher Secondary Schools',
   'score': 0.8724746704101562},
  {'name': 'Mr. Moto Takes a Chance', 'score': 0.8724746704101562},
  {'name': 'The Da Vinci Code (film)', 'score': 0.8724746704101562},
  {'name': 'Every Day I Have to Cry', 'score': 0.7815757989883423},
  {'name': 'One Hundred Nails', 'score': 0.7579530477523804},
  {'name': 'Charge It to Me', 'score': 0.0}],
 'Méditerranée': [{'name': 'Méditerranée (1963 film)',
   'score': 2.2738592624664307}],
 'Film': [{'name': 'Fire Down Below (1957 film)', 'score': 1.8455064296722412},
  {'name': 'Musa (film)', 'score': 1.4944583177566528},
  {'name': 'Blücher (film)', 'score': 1.4944583177566528},
  {'name

In [171]:
res_busqueda_embeddings

[<Record name='Méditerranée (1963 film)' score=0.8802040815353394>,
 <Record name='Alsino and the Condor' score=0.7534630298614502>,
 <Record name='Jean-Daniel Pollet' score=0.7435126304626465>,
 <Record name='Escape to France' score=0.7428261637687683>,
 <Record name='1960' score=0.7390859127044678>]

In [155]:
entidades_filtradas = []

Todas las entidades exactas

In [156]:
entidades_filtradas = entis_exactas

Busquedas parcial y fuzzy solo si score > 2

In [ ]:
filtrado_fuzzy=[]
for k,v in res_busqueda_fuzzy.items():
    for ent in v:
        if ent['score'] > 2:
            filtrado_fuzzy.append(ent['name'])

In [ ]:
filtrado_parcial=[]
for k,v in res_busqueda_parcial.items():
    for ent in v:
        if ent['score'] > 2:
            filtrado_parcial.append(ent['name'])

In [ ]:
filtrado_fuzzy

['Move (1970 film)',
 'Méditerranée (1963 film)',
 'Move (1970 film)',
 '1960',
 'Méditerranée (1963 film)',
 '1960']

In [ ]:
filtrado_parcial

['Move (1970 film)',
 'Méditerranée (1963 film)',
 'Move (1970 film)',
 'Méditerranée (1963 film)']

Conteo de cuantas veces aparecen entre las 2

In [ ]:
from collections import Counter

In [ ]:
conteo_fuzzy_parcial = Counter(filtrado_fuzzy + filtrado_parcial)

In [ ]:
conteo_fuzzy_parcial

Counter({'Move (1970 film)': 4, 'Méditerranée (1963 film)': 4, '1960': 2})

In [188]:
conteo_fuzzy_parcial.most_common(3)

[('Move (1970 film)', 4), ('Méditerranée (1963 film)', 4), ('1960', 2)]

In [189]:
entidades_mas_comunes = [k for k,v in conteo_fuzzy_parcial.most_common(3)]


In [190]:
entidades_mas_comunes

['Move (1970 film)', 'Méditerranée (1963 film)', '1960']

Añadir a una lista las 2 con mayor score de los embeddings

In [191]:
res_busqueda_embeddings

[<Record name='Méditerranée (1963 film)' score=0.8802040815353394>,
 <Record name='Alsino and the Condor' score=0.7534630298614502>,
 <Record name='Jean-Daniel Pollet' score=0.7435126304626465>,
 <Record name='Escape to France' score=0.7428261637687683>,
 <Record name='1960' score=0.7390859127044678>]

In [195]:
embeddings_filt = [ent["name"] for ent in res_busqueda_embeddings[:2]]
embeddings_filt

['Méditerranée (1963 film)', 'Alsino and the Condor']

In [196]:
entidades_finales = list(set(entidades_mas_comunes + embeddings_filt))
entidades_finales

['1960',
 'Move (1970 film)',
 'Méditerranée (1963 film)',
 'Alsino and the Condor']

## FUNCIONES

In [200]:
def filtrado_fuzzy_parcial(results_fuzzy, results_parciales):
    filtrado_fuzzy=[]
    for k,v in results_fuzzy.items():
        for ent in v:
            if ent['score'] > 2:
                filtrado_fuzzy.append(ent['name'])
                
    filtrado_parcial=[]
    for k,v in results_parciales.items():
        for ent in v:
            if ent['score'] > 2:
                filtrado_parcial.append(ent['name'])
    conteo_fuzzy_parcial = Counter(filtrado_fuzzy + filtrado_parcial)
    entidades_mas_comunes = [k for k,v in conteo_fuzzy_parcial.most_common(3)]
    return entidades_mas_comunes

In [198]:
def extraer_top_k_entities(query_result, k):
    entities_found = []
    for entity in range(k):
        entities_found.append(query_result[entity]['name'])
    
    return entities_found

In [ ]:
def union_entidades(entis_exactas, entidades_text_index, entidades_embeddings):
    entidades_finales = list(set(entis_exactas + entidades_text_index + entidades_embeddings))
    return entidades_finales

In [201]:
entidades_text_index = filtrado_fuzzy_parcial(res_busqueda_fuzzy, res_busqueda_parcial)

In [204]:
entidades_embeddings = extraer_top_k_entities(res_busqueda_embeddings, 2)

In [ ]:
entidades_finales = union_entidades(entis_exactas, entidades_text_index, entidades_embeddings)

In [206]:
entidades_finales

['1960',
 'Move (1970 film)',
 'Méditerranée (1963 film)',
 'Alsino and the Condor']